In [ ]:
import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from dataset import TimepieceDataset

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

## CNN

In [ ]:
# ===================== 1. Define the model (CNN) =====================
class DigitalClockNet(nn.Module):
    def __init__(self):
        super(DigitalClockNet, self).__init__()
        
        # First convolutional block: basic feature extraction (edges, corners)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Make it 64x64
        )
        
        # Second convolutional block: more complex features
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 32x32
        )
        
        # Third convolutional block: high-level features
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 16x16
        )

        # Fully connected layers for regression
        # Input size: 128 channels * 16 * 16 spatial dimensions
        self.fc = nn.Sequential( 
            nn.Flatten(), # Flatten the tensor
            nn.Linear(128 * 16 * 16, 512), # Fully connected layer 
            nn.ReLU(),
            nn.Dropout(0.5), # Reduce overfitting
            nn.Linear(512, 3) # Output: (h, m, s)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x # tensor of shape (batch_size, 3)

In [ ]:
# ===================== 2. Training function =====================
def train_model(data_dir, num_epochs=10, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # Data transformations and loaders
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])
    
    train_dataset = TimepieceDataset(data_dir, subset='train', transform=transform)
    test_dataset = TimepieceDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Making the model 
    model = DigitalClockNet().to(device)
    
    # Loss Function: MSE (Mean Squared Error) for regression
    criterion = nn.MSELoss()
    
    # Optimizer: Adam optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting training...")
    
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device) # Normalized (h, m, s)
            
            # 1. Forward pass
            outputs = model(images)
            
            # 2. Calculate loss
            loss = criterion(outputs, labels)
            
            # 3. Backward pass and optimization
            optimizer.zero_grad() # Clear gradients
            loss.backward() # Backpropagation
            optimizer.step() # Update weights
            
            running_loss += loss.item() # Update running loss

        avg_train_loss = running_loss / len(train_loader) 

        # Calculate test loss
        model.eval()
        test_loss = 0.0
        with torch.no_grad(): # No gradient calculation during evaluation
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save the best model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            torch.save(model.state_dict(), "checkpoints/digital_reader_best.pth")
            print("  -> Saved best model!")

In [ ]:
# ===================== 3. Run =====================
if __name__ == "__main__":
    # Start training with specified data directory and epochs 
    train_model(data_dir="../data", num_epochs=100)

In [ ]:
def denormalize_time(tensor_time):
    h = int(tensor_time[0].item() * 23)
    m = int(tensor_time[1].item() * 59)
    s = int(tensor_time[2].item() * 59)
    return f"{h:02d}:{m:02d}:{s:02d}"

def evaluate():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Loading the test dataset for evaluation
    test_dataset = TimepieceDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockNet().to(device) # Load the trained model
    try:
        model.load_state_dict(torch.load("checkpoints/digital_reader_best.pth", map_location=device))
        print("CNN Model loaded successfully!")
    except FileNotFoundError:
        print("Error: Checkpoint not found.")
        return

    model.eval() # Set model to evaluation mode

    print("\n--- Visual Evaluation CNN ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    with torch.no_grad(): # No gradient calculation during evaluation
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            # Calculate loss for reporting
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.4f}")

if __name__ == "__main__":
    evaluate()

## Res-Net

In [ ]:
# ===================== 1. Define the ResNet Model =====================
class DigitalClockResNet(nn.Module):
    def __init__(self):
        super(DigitalClockResNet, self).__init__()
        
        # Load a pre-trained ResNet18 model
        # "DEFAULT" weights mean it was trained on ImageNet (millions of images)
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # We need to replace the last layer (fc) because ResNet outputs 1000 classes by default.
        # We want it to output 3 numbers (Hour, Minute, Second).
        
        num_features = self.model.fc.in_features # usually 512 for ResNet18
        
        self.model.fc = nn.Sequential(
            nn.Linear(num_features, 256), # Compress features
            nn.ReLU(),
            nn.Dropout(0.3),              # Prevent overfitting [cite: 46]
            nn.Linear(256, 3)             # Output: (h, m, s)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# ===================== 2. ResNet Training Function =====================
def train_resnet_model(data_dir, num_epochs=20, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # TRANSFORMATIONS: ResNet requires 224x224 and specific normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        # Standard ImageNet normalization (Mean, Std)
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Load Data
    # Note: Ensure data_dir points to the folder containing 'train' and 'test' folders
    train_dataset = TimepieceDataset(data_dir, subset='train', transform=transform)
    test_dataset = TimepieceDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize ResNet
    model = DigitalClockResNet().to(device)
    
    # Loss and Optimizer
    criterion = nn.MSELoss()
    # Lower learning rate (0.0001) is better for fine-tuning
    optimizer = optim.Adam(model.parameters(), lr=learning_rate) 

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting ResNet training...")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device)
            
            # Forward -> Loss -> Backward
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader) 

        # Validation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save Best Model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            # Save with a distinct name so we don't overwrite the CNN
            torch.save(model.state_dict(), "checkpoints/digital_resnet_best.pth")
            print("  -> Saved best ResNet model!")

# Run the training
# Make sure data_dir is correct (e.g., "./data" or "../data")
if __name__ == "__main__":
    train_resnet_model(data_dir="../data", num_epochs=100)

In [ ]:
def evaluate_resnet():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # MUST MATCH TRAINING TRANSFORMS
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    test_dataset = TimepieceDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockResNet().to(device)
    
    try:
        # Load the ResNet checkpoint
        model.load_state_dict(torch.load("checkpoints/digital_resnet_best.pth", map_location=device))
        print("ResNet Model loaded successfully!")
    except FileNotFoundError:
        print("Error: ResNet Checkpoint not found.")
        return

    model.eval()

    print("\n--- Visual Evaluation (ResNet) ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    def denormalize_time(tensor_time):
        # Clamp ensures values don't go below 0 or above 1 due to minor noise
        tensor_time = torch.clamp(tensor_time, 0, 1) 
        h = int(tensor_time[0].item() * 23)
        m = int(tensor_time[1].item() * 59)
        s = int(tensor_time[2].item() * 59)
        return f"{h:02d}:{m:02d}:{s:02d}"

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.6f}")

if __name__ == "__main__":
    evaluate_resnet()

In [ ]:
# ===================== MODEL =====================

class DigitalClockClassifier(nn.Module):
    """
    ResNet18 backbone with 3 independent classification heads:
      - hour_head:   24 classes  (0–23)
      - minute_head: 60 classes  (0–59)
      - second_head: 60 classes  (0–59)

    Why classification instead of regression?
    Regression treats 12:59 and 13:00 as "close" in loss space,
    even though they are very different clock positions.
    Classification treats each digit combination as an independent class,
    so the model can learn crisp decision boundaries.
    """
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        feat = base.fc.in_features          # 512 for ResNet18
        base.fc = nn.Identity()             # Remove the original head
        self.backbone = base

        # Shared bottleneck (helps all 3 heads)
        self.bottleneck = nn.Sequential(
            nn.Linear(feat, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.hour_head   = nn.Linear(256, 24)
        self.minute_head = nn.Linear(256, 60)
        self.second_head = nn.Linear(256, 60)

    def forward(self, x):
        f = self.backbone(x)
        f = self.bottleneck(f)
        return self.hour_head(f), self.minute_head(f), self.second_head(f)

    def predict_time(self, x):
        """Returns integer (h, m, s) — use this at inference time."""
        h_logits, m_logits, s_logits = self.forward(x)
        h = h_logits.argmax(dim=1)
        m = m_logits.argmax(dim=1)
        s = s_logits.argmax(dim=1)
        return h, m, s

In [ ]:
def train(data_dir='../data', num_epochs=80, batch_size=32, lr=1e-4):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        # More aggressive augmentation — helps distinguish similar digits
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # slight shift
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = TimepieceDataset(data_dir, subset='train', transform=transform)
    test_ds  = TimepieceDataset(data_dir, subset='test',  transform=transform_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    model     = DigitalClockClassifier().to(device)
    # Label smoothing helps when similar digits are being confused (e.g. 52 vs 54)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Warmup for 5 epochs then cosine decay — prevents early bad local minima
    def lr_lambda(epoch):
        if epoch < 5:
            return (epoch + 1) / 5   # linear warmup
        # cosine decay from epoch 5 to num_epochs
        progress = (epoch - 5) / (num_epochs - 5)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    import math
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    os.makedirs('checkpoints', exist_ok=True)
    best_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            imgs   = batch['digital_img'].to(device)
            labels = batch['original_time'].to(device)
            h_true, m_true, s_true = labels[:,0], labels[:,1], labels[:,2]

            h_pred, m_pred, s_pred = model(imgs)
            # Weight minute and second loss higher — they are harder (60 classes vs 24)
            loss = (criterion(h_pred, h_true) +
                    2.0 * criterion(m_pred, m_true) +
                    2.0 * criterion(s_pred, s_true))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        model.eval()
        correct_h = correct_m = correct_s = correct_all = total = 0
        with torch.no_grad():
            for batch in test_loader:
                imgs   = batch['digital_img'].to(device)
                labels = batch['original_time'].to(device)
                h_true, m_true, s_true = labels[:,0], labels[:,1], labels[:,2]
                h_p, m_p, s_p = model.predict_time(imgs)
                correct_h   += (h_p == h_true).sum().item()
                correct_m   += (m_p == m_true).sum().item()
                correct_s   += (s_p == s_true).sum().item()
                correct_all += ((h_p == h_true) & (m_p == m_true) & (s_p == s_true)).sum().item()
                total += imgs.size(0)

        acc_all = 100 * correct_all / total
        print(f"Epoch [{epoch+1:>3}/{num_epochs}]  "
              f"Loss: {total_loss/len(train_loader):.4f}  lr: {current_lr:.5f}  |  "
              f"H: {100*correct_h/total:.0f}%  "
              f"M: {100*correct_m/total:.0f}%  "
              f"S: {100*correct_s/total:.0f}%  |  "
              f"ALL: {acc_all:.1f}%")

        if acc_all > best_acc:
            best_acc = acc_all
            torch.save(model.state_dict(), 'checkpoints/digital_reader_best.pth')
            print(f"  -> Saved! Best: {best_acc:.1f}%")

In [ ]:
# ===================== EVALUATION =====================

def evaluate(data_dir='./clock_dataset'):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    test_ds = TimepieceDataset(data_dir, subset='test', transform=transform)
    loader  = DataLoader(test_ds, batch_size=1, shuffle=True, num_workers=0)

    model = DigitalClockClassifier().to(device)
    model.load_state_dict(torch.load('checkpoints/digital_reader_best.pth', map_location=device))
    model.eval()

    print(f"\n{'Actual':<12} {'Predicted':<12} {'Correct?'}")
    print("-" * 40)

    errors = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= 20: break
            imgs = batch['digital_img'].to(device)
            h_true, m_true, s_true = (batch['original_time'][:, j].item() for j in range(3))

            h_pred, m_pred, s_pred = model.predict_time(imgs)
            h_p, m_p, s_p = h_pred.item(), m_pred.item(), s_pred.item()

            actual = f"{h_true:02d}:{m_true:02d}:{s_true:02d}"
            pred   = f"{h_p:02d}:{m_p:02d}:{s_p:02d}"
            ok     = "OK" if actual == pred else "WRONG"
            print(f"{actual:<12} {pred:<12} {ok}")

            # Show first failure visually
            if ok == "WRONG" and len(errors) == 0:
                errors.append((imgs.cpu(), actual, pred))

    if errors:
        img_t, actual, pred = errors[0]
        plt.figure(figsize=(4, 4))
        plt.imshow(img_t[0].permute(1, 2, 0).clamp(0, 1))
        plt.title(f"Failure: actual={actual}  pred={pred}", color='red')
        plt.axis('off')
        plt.tight_layout()
        print("\nSaved failure example to reader_failure.png")

In [ ]:
train(data_dir='./clock_dataset', num_epochs=80)


In [ ]:
evaluate()